In [3]:
import pandas as pd
import random

In [7]:
#Read Orders sheet
orders= pd.read_excel("Supply_Chain_Data.xlsx",sheet_name="Orders")

In [6]:
import pandas as pd
import random

# Read Orders sheet
orders = pd.read_excel(
    "Supply_Chain_Data.xlsx",
    sheet_name="Orders"
)

# Convert Order Date to date format
orders["Order Date"] = pd.to_datetime(
    orders["Order Date"]
)

# Create Forecast Month column (2025-01, 2025-02, etc.)
orders["ForecastMonth"] = orders[
    "Order Date"
].dt.strftime("%Y-%m")

# Monthly demand by product
forecast_df = (
    orders.groupby(
        ["ForecastMonth", "Product ID"]
    )["Quantity"]
    .sum()
    .reset_index()
)

# Rename Quantity column
forecast_df.rename(
    columns={"Quantity": "ActualDemand"},
    inplace=True
)

# Generate forecast demand
forecast_df["ForecastDemand"] = (
    forecast_df["ActualDemand"]
    .apply(
        lambda x: round(
            x * random.uniform(0.9, 1.1)
        )
    )
)

# Forecast Accuracy %
forecast_df["ForecastAccuracy"] = (
    (
        1
        - abs(
            forecast_df["ActualDemand"]
            - forecast_df["ForecastDemand"]
        )
        / forecast_df["ActualDemand"]
    )
    * 100
).round(2)

# Demand Variance
forecast_df["DemandVariance"] = (
    forecast_df["ActualDemand"]
    - forecast_df["ForecastDemand"]
)

# Save to Excel
with pd.ExcelWriter(
    "Supply_Chain_Data.xlsx",
    mode="a",
    engine="openpyxl",
    if_sheet_exists="replace"
) as writer:

    forecast_df.to_excel(
        writer,
        sheet_name="Forecast",
        index=False
    )

print("Forecast sheet created successfully!")

Forecast sheet created successfully!
